# 02 · SAFT molecules, SMILES and group parameters

**Requirements:** SAFT/JAX and RDKit.

Run this notebook from top to bottom in the environment where ThermoWeave is installed. All examples are self-contained unless explicitly marked optional. See the [course index](../tutorials.md) and [theory guide](../theory.md).

<a class="md-button" href="02_saft_molecules_and_groups.ipynb" download>Download notebook</a>


SAFT-γ Mie uses a catalogue of thermodynamic groups. These include
 composite units such as `CH2OH`, `aCNH2` and `aCCOO`; a generic list of
 chemical functional groups is not interchangeable with this catalogue.
 The report below shows counts, matched atoms, SMARTS rules and references.
 Atom labels in the printed SMILES are 1-based; stored `atom_indices` are 0-based.

In [1]:
from thermoweave import SAFTgammaMie
from thermoweave.saft import assign_groups, GroupDatabase, missing_interactions, AssignmentError
assignment = assign_groups('CCO', name='ethanol')
assert assignment.counts == {'CH3': 1, 'CH2OH': 1}

SAFT-gamma-Mie automatic assignment: ethanol
  SMILES: CCO
  Formula: C2H6O
  Groups: {'CH2OH': 1, 'CH3': 1}
  Atom-labelled SMILES (labels are 1-based): [CH3:1][CH2:2][OH:3]
  Method: specific SMARTS groups first; complete, non-overlapping atom coverage.
  CH2OH x1:
    atoms 2:C(H2), 3:O(H1)
    rule: primary alcohol carbon plus hydroxyl; SMARTS [C;!R;H2;X4]-[O;H1;X2]
    parameters: 10.1021/acs.jced.0c00746
  CH3 x1:
    atoms 1:C(H3)
    rule: acyclic saturated methyl carbon; SMARTS [C;!R;H3;X4]
    parameters: 10.1021/acs.jced.0c00746
  Coverage: 3/3 heavy atoms; each atom assigned once; elemental composition including H checked.
  Unlike interaction coverage is checked separately with missing_interactions().


In [2]:
model = SAFTgammaMie({'water': 'O', 'ethanol': 'CCO'}, pressure_Pa=101325.)
manual = SAFTgammaMie({'water': {'H2O': 1}, 'ethanol': {'CH3': 1, 'CH2OH': 1}},
                     pressure_Pa=101325.)
assert model.groups == manual.groups
# Mixed input works too; only automatic components produce assignment reports.
mixed = SAFTgammaMie({'water': 'O', 'ethanol': {'CH3': 1, 'CH2OH': 1}},
                    pressure_Pa=101325., report_assignment=False)
print('Retained automatic reports:', list(mixed.assignments))

SAFT-gamma-Mie automatic assignment: water
  SMILES: O
  Formula: H2O
  Groups: {'H2O': 1}
  Atom-labelled SMILES (labels are 1-based): [OH2:1]
  Method: specific SMARTS groups first; complete, non-overlapping atom coverage.
  H2O x1:
    atoms 1:O(H2)
    rule: whole water molecule; SMARTS [O;H2;X2]
    parameters: 10.1021/acs.jced.0c00746
  Coverage: 1/1 heavy atoms; each atom assigned once; elemental composition including H checked.
  Unlike interaction coverage is checked separately with missing_interactions().


SAFT-gamma-Mie automatic assignment: ethanol
  SMILES: CCO
  Formula: C2H6O
  Groups: {'CH2OH': 1, 'CH3': 1}
  Atom-labelled SMILES (labels are 1-based): [CH3:1][CH2:2][OH:3]
  Method: specific SMARTS groups first; complete, non-overlapping atom coverage.
  CH2OH x1:
    atoms 2:C(H2), 3:O(H1)
    rule: primary alcohol carbon plus hydroxyl; SMARTS [C;!R;H2;X4]-[O;H1;X2]
    parameters: 10.1021/acs.jced.0c00746
  CH3 x1:
    atoms 1:C(H3)
    rule: acyclic saturated methyl carbon; SMARTS [C;!R;H3;X4]
    parameters: 10.1021/acs.jced.0c00746
  Coverage: 3/3 heavy atoms; each atom assigned once; elemental composition including H checked.
  Unlike interaction coverage is checked separately with missing_interactions().


Retained automatic reports: ['water']


## Explicit extension tables and coverage
 The Wehbe extension adds aromatic ester/amine groups. The Tenberg extension
 adds pyridine-type nitrogen. Parameters stay tied to their sources.
 Group assignment and interaction-parameter coverage are separate checks.

In [3]:
db = GroupDatabase.default('wehbe2023', 'tenberg2025')
benzocaine = assign_groups('CCOC(=O)c1ccc(N)cc1', name='benzocaine', database=db)
components = {'benzocaine': benzocaine.counts, 'ethanol': assignment.counts}
print('Unlike-parameter gaps:', missing_interactions(components, db))
# These gaps are real: a recognized molecule is not necessarily fully parameterized.

SAFT-gamma-Mie automatic assignment: benzocaine
  SMILES: CCOC(=O)c1ccc(N)cc1
  Formula: C9H11NO2
  Groups: {'CH2': 1, 'CH3': 1, 'aCCOO': 1, 'aCH': 4, 'aCNH2': 1}
  Atom-labelled SMILES (labels are 1-based): [CH3:1][CH2:2][O:3][C:4](=[O:5])[c:6]1[cH:7][cH:8][c:9]([NH2:10])[cH:11][cH:12]1
  Method: specific SMARTS groups first; complete, non-overlapping atom coverage.
  CH2 x1:
    atoms 2:C(H2)
    rule: acyclic saturated methylene carbon; SMARTS [C;!R;H2;X4]
    parameters: 10.1021/acs.jced.0c00746
  CH3 x1:
    atoms 1:C(H3)
    rule: acyclic saturated methyl carbon; SMARTS [C;!R;H3;X4]
    parameters: 10.1021/acs.jced.0c00746
  aCCOO x1:
    atoms 3:O(H0), 4:C(H0), 5:O(H0), 6:C(H0)
    rule: aromatic carbon plus ester C(=O)O; alkyl carbon excluded; SMARTS [c;H0;r6]-[C;X3](=[O;X1])-[O;H0;X2]-[C;X4]
    parameters: Wehbe, PhD thesis, Imperial College London (2023), doi:10.25560/103095, Tables 4.2-4.4
  aCH x4:
    atoms 7:C(H1)
    atoms 8:C(H1)
    atoms 11:C(H1)
    atoms 12:C(H1)
 

Unlike-parameter gaps: {'mie': [('CH2OH', 'aCCOO'), ('CH2OH', 'aCNH2')], 'association': {('CH2OH', 'aCCOO'): 'no association', ('CH2OH', 'aCH'): 'no association', ('CH2OH', 'aCNH2'): 'combining rule', ('aCH', 'aCNH2'): 'no association'}}


In [4]:
for name, smiles in [('dimethyl ether', 'COC'), ('nicotinamide', 'NC(=O)c1cccnc1')]:
    try:
        assign_groups(smiles, name=name, database=db)
    except AssignmentError as exc:
        print(f'{name}: {exc}')
    else:
        raise AssertionError('Expected an uncovered rule or missing group parameters')

dimethyl ether: dimethyl ether: no SAFT assignment rule for atom(s) 2:O in 'COC'. Use manual groups or add a FragmentationRule.
nicotinamide: nicotinamide: identified 'aCCONH2', but its parameters are missing: "unknown SAFT-gamma-Mie group 'aCCONH2'"


Nicotinamide's `aCCONH2` rule requires an added, parameterized aromatic
 primary-amide group. Thesis-specific fits are not bundled. Unsupported or
 ambiguous structures raise an error; the mapper does not invent groups or
 parameters. Manual dictionaries remain available. Explicit custom
 `FragmentationRule` objects can extend `assign_groups`; see the API guide.

 **Try:** assign 1-butanol (`CCCCO`), isopropanol (`CC(O)C`) and tert-butanol
 (`CC(O)(C)C`) and inspect which atoms enter the alcohol group.